# 10 - FFSP: a small stochastic finite-fault source

The **Finite Fault Stochastic Process** (FFSP) source model generates a
kinematic rupture on a finite fault: each subfault gets a slip, rupture time,
rise time, peak time and a (perturbed) strike/dip/rake.

Code: (c) 2005 Pengcheng Liu, modifications by Chen Ji (2020); method from
Liu, Archuleta & Hartzell (2006). Adapted for use within ShakerMaker.

Here we build a deliberately **small and fast** source (16 x 8 subfaults, a
single realization with fixed seeds) so the Fortran kernel finishes quickly,
then visualize:

1. the **spatial distribution** of a subfault field (`plot_spacial_distribution`),
2. the **source time function** (`plot_source_time_function`).

Each figure is saved as a `.png` in this folder.

In [ ]:
import matplotlib.pyplot as plt
from shakermaker.crustmodel import CrustModel
from shakermaker.ffspsource import FFSPSource

## Crust model

A 4-layer crust (slow sediments over a faster half-space).
`add_layer(thickness_km, vp, vs, rho, qa, qb)`.

In [ ]:
crust = CrustModel(4)
crust.add_layer(0.200,  1.32, 0.75, 2.40, 1000., 1000.)
crust.add_layer(0.800,  2.75, 1.57, 2.50, 1000., 1000.)
crust.add_layer(14.500, 5.50, 3.14, 2.50, 1000., 1000.)
crust.add_layer(0.000,  7.00, 4.00, 2.67, 1000., 1000.)
print(crust)

## Preview the crust before running FFSP
The FFSP source uses this crustal model. We plot its layers and velocity profile before generating realizations. (The finite-fault source itself is visualized after `run()` with the FFSP plotting methods below.)

In [ ]:
import matplotlib.pyplot as plt
crust.plot()
plt.gcf().savefig("crust_layers.png", dpi=150, bbox_inches="tight")
crust.plot_profile()
plt.gcf().savefig("crust_velocity_profile.png", dpi=150, bbox_inches="tight")

## Build the FFSP source

Small grid (`nsubx=16`, `nsuby=8`), a single realization
(`id_ran1=id_ran2=1`) and fixed `seeds` keep this reproducible and fast.

In [ ]:
source = FFSPSource(
    id_sf_type=8, freq_min=0.01, freq_max=24.0,
    fault_length=30.0, fault_width=16.0,
    x_hypc=15.0, y_hypc=8.0, depth_hypc=8.0,
    xref_hypc=0.0, yref_hypc=0.0,
    magnitude=6.0, fc_main_1=0.09, fc_main_2=3.0,
    rv_avg=3.0,
    ratio_rise=0.3,
    strike=358.0, dip=40.0, rake=113.0,
    pdip_max=15.0, prake_max=30.0,
    nsubx=16, nsuby=8,
    nb_taper_trbl=[5, 5, 5, 5],
    seeds=[52, 448, 4446],
    id_ran1=1, id_ran2=1,
    angle_north_to_x=0.0,
    is_moment=3,
    crust_model=crust,
    output_name="FFSP_OUTPUT",
    verbose=True,
)

## Run the rupture generator (Fortran kernel)

**This is the heavy cell** — `source.run()` calls the FFSP Fortran kernel and
may take a while even for this small grid. It returns the best realization's
subfault dictionary and stores it on the object.

In [ ]:
subfaults = source.run()   # <-- RUN cell (slow)
print("npts:", subfaults["npts"], " slip max:", subfaults["slip"].max())

## Spatial distribution of slip

`plot_spacial_distribution` maps a per-subfault field over the fault plane,
with rupture-time contours and the hypocenter. We pass `save_fig=True` with a
`model_name`; the method writes `<model_name>_*.png`. We also re-save via
`plt.gcf().savefig` to guarantee a `.png` in this folder.

In [ ]:
source.plot_spacial_distribution(field="slip", show_contours=True,
                                 show_hypocenter=True)
plt.gcf().savefig("ffsp_slip_distribution.png", dpi=150, bbox_inches="tight")

## Source time function

`plot_source_time_function` plots the moment-rate STF of the best realization.

In [ ]:
source.plot_source_time_function()
plt.gcf().savefig("ffsp_source_time_function.png", dpi=150, bbox_inches="tight")